# OTTO · Feature Round 01
## Observed-intent support and evidence timing

**Purpose:** explain 48 proposed features before committing compute to their value. This notebook contains an executed mathematical illustration, **not an OTTO model result**. Run the real experiment in `01_run_and_analyze.ipynb`.

Your uploaded readiness report passed. No reset, clone, raw-data download or environment installation is needed. The existing candidate pools and certified historical graphs stay unchanged.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import plotly.graph_objects as go
from IPython.display import display
ROOT = Path.cwd()
if not (ROOT / 'intent_features.py').exists():
    ROOT = Path.home() / 'otto_feature_round01'
sys.path.insert(0, str(ROOT))
from intent_features import NAMES, SUPPORT, TIMING, seed_pools, summarize_support
assert len(NAMES) == 48 and len(SUPPORT) == len(TIMING) == 24
print('Proposed features:', len(NAMES))
print('24 support + 24 timing; no model score has been generated.')

Proposed features: 48
24 support + 24 timing; no model score has been generated.


### Mechanism, not column multiplication

Two graph families × three action channels × two observed-item pools × four summaries.

**Support:** what fraction of other observed items supports a candidate, and how spread out is the evidence?

**Timing:** how old is the support, weighted by affinity, and how old is its strongest source?

A candidate's self-edge is excluded. The pending-cart pool means “cart observed after latest observed order”; it is not actual cart inventory and does not identify abandonment.

In [2]:
# Explicitly synthetic affinities solely for checking feature meaning.
seeds = np.array([11, 12, 13], dtype=np.int64)
candidates = np.array([101, 102, 103, 104], dtype=np.int64)
age_hours = np.array([0.0, 1.0, 5.0])
affinity = np.array([[3.0, 1.0, 0.0, 0.0],
                     [0.0, 1.0, 0.0, 0.0],
                     [0.0, 1.0, 3.0, 0.0]])
summary = summarize_support(affinity, seeds, candidates, age_hours)
labels = ['One recent supporter', 'Three equal supporters', 'One older supporter', 'No support']
fig = go.Figure(go.Heatmap(x=labels, y=['0 hours old','1 hour old','5 hours old'], z=affinity))
fig.update_layout(title='Illustration only — candidate support from other products',
                  xaxis_title='Hypothetical candidate', yaxis_title='Observed seed item age')
fig.show()

In [3]:
fig = go.Figure()
fig.add_trace(go.Bar(name='Support coverage', x=labels, y=summary[:,0]))
fig.add_trace(go.Bar(name='Normalized evidence entropy', x=labels, y=summary[:,1]))
fig.update_layout(title='Illustration only — equal total affinity need not mean equal support',
                  barmode='group', yaxis_title='Feature value, not predictive performance')
fig.show()
assert np.allclose(summary[:,0], [1/3, 1, 1/3, 0])
assert np.allclose(summary[:,1], [0, 1, 0, 0])

In [4]:
fig = go.Figure()
fig.add_trace(go.Bar(name='Affinity-weighted mean log age', x=labels, y=summary[:,2]))
fig.add_trace(go.Bar(name='Strongest-support log age', x=labels, y=summary[:,3]))
fig.update_layout(title='Illustration only — distinguish current from older supporting evidence',
                  barmode='group', yaxis_title='log(1 + hours)')
fig.show()
assert summary[3].tolist() == [0, 0, 0, 0]

### Observe the action history correctly

The last observed order clears a previous cart state; a later observed cart reopens it. A later click does not clear it. No future label is part of this calculation.

In [5]:
aids = np.array([11,11,11,12,12,13], dtype=np.int64)
kinds = np.array([1,2,1,1,2,0], dtype=np.int64)
ts = np.arange(6, dtype=np.int64) * 1000 + 1660687201000
pools = seed_pools(aids, ts, kinds, int(ts[-1]))
assert pools['pending_cart'][0].tolist() == [11]
print('Pending-cart seeds:', pools['pending_cart'][0].tolist())
print('Item 11 re-entered the cart after an observed order.')
print('Item 12 had a later order and is excluded. Item 13 was only clicked.')

Pending-cart seeds: [11]
Item 11 re-entered the cart after an observed order.
Item 12 had a later order and is excluded. Item 13 was only clicked.


### What would count as evidence?

The actual study preserves 1,024 fitting sessions and 400 candidates each. It reuses the saved control at 134 features, adds support alone (158), timing alone (158), and both (182). Only 18 new objective/fold models are needed.

The two forward folds retain the prior six-hour embargo and censor training targets before negative sampling. Compare pooled Recall@20, each objective, both time-fold signs, and paired uncertainty. Removing either new group from the full 48-feature arm provides a matched group ablation.

The stage must not report a Kaggle improvement, promote a feature automatically, or erase negative findings. The prior control's approximately 0.484737 is internal fitting-only performance, not the 0.56842 submitted score.

### Research basis

[First-place OTTO writeup](https://www.kaggle.com/competitions/otto-recommender-system/writeups/mrkmakr-1st-place-solution) and [third-place author's implementation](https://github.com/TheoViel/kaggle_otto_rs) motivate weighted candidate–session relationships. These sources do **not** establish the efficacy of our exact formulas. See `RESEARCH_PLAN.md` for definitions, availability, limitations, next rounds, and stopping rules.